# Módulo 05 · Aula 03 — MongoDB

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O catálogo muda toda semana. Entrou uma linha de cadeiras gamer: precisa de altura regulável, tipo de espuma e peso suportado. Semana passada foram fones, com impedância e tipo de conexão. Não dá para pedir uma migração de banco a cada categoria nova."*
> — Gerente de Produto

Você viu no 05_01 que JSONB resolve boa parte disso. Esta aula responde: **quando vale a pena ir além e usar um banco de documentos de verdade?**

## O que você vai aprender aqui

| # | Tópico | Resolve |
|---|--------|---------|
| 1 | Documento vs relação | O modelo mental |
| 2 | Quando NÃO usar MongoDB | ⚠️ A parte mais importante |
| 3 | BSON e tipos | O formato interno |
| 4 | PyMongo — CRUD | Operações do dia a dia |
| 5 | Consultas e operadores | `$gte`, `$in`, `$regex`, aninhamento |
| 6 | Atualização parcial | `$set`, `$inc`, `$push`, upsert |
| 7 | Modelagem: embutir ou referenciar | **A decisão central** |
| 8 | **Pipeline de agregação** | O `GROUP BY` do Mongo |
| 9 | Índices | Inclusive compostos e de texto |
| 10 | Transações e consistência | O que mudou desde a fama antiga |

> ▶️ **Este notebook roda em qualquer máquina.** Se houver um MongoDB de pé, ele usa; se não, usa **mongomock** — que implementa a API do PyMongo em memória. O código é idêntico.

In [ ]:
import os
import socket
import subprocess
import sys


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


def _porta_aberta(host, porta, timeout=0.7):
    try:
        with socket.create_connection((host, porta), timeout=timeout):
            return True
    except OSError:
        return False


TEM_MONGO = _porta_aberta(os.getenv("MONGOHOST", "localhost"),
                          int(os.getenv("MONGOPORT", "27017")))

# ASCENDING e DESCENDING são apenas 1 e -1 — funcionam nos dois modos.
ASCENDING, DESCENDING = 1, -1

if TEM_MONGO and _garantir("pymongo"):
    from pymongo import MongoClient
    cliente = MongoClient(os.getenv("ATLAS_MONGO_URI", "mongodb://localhost:27017"))
    MODO = "real"
else:
    _garantir("mongomock")
    import mongomock
    cliente = mongomock.MongoClient()
    MODO = "mongomock"

db = cliente["atlas_catalogo"]

# Limpa para o notebook ser reexecutável
for colecao in db.list_collection_names():
    db.drop_collection(colecao)

print(f"✅ Conectado — modo: {MODO}")
if MODO == "mongomock":
    print()
    print("   Para usar o MongoDB de verdade:")
    print("       cd projeto_Atlas && docker compose up -d mongo")
    print("   Depois reinicie o kernel.")
    print()
    print("   ⚠️ O mongomock cobre a maior parte da API, mas não tudo.")
    print("      Transações e alguns operadores exigem o servidor real.")

In [ ]:
# Função auxiliar para exibir documentos de forma legível
import json


def mostrar(documentos, titulo="", limite=10):
    """Imprime documentos formatados."""
    if titulo:
        print(f"── {titulo} ──")
    docs = list(documentos)
    for i, d in enumerate(docs[:limite]):
        print(json.dumps(d, ensure_ascii=False, indent=2, default=str))
        if i < min(len(docs), limite) - 1:
            print()
    if len(docs) > limite:
        print(f"... e mais {len(docs) - limite}")
    if not docs:
        print("(nenhum documento)")
    print(f"[{len(docs)} documento(s)]\n")
    return docs


def tabela(documentos, colunas, titulo=""):
    """Imprime documentos como tabela — mais compacto que JSON."""
    docs = list(documentos)
    if titulo:
        print(f"── {titulo} ──")
    if not docs:
        print("(nenhum documento)\n")
        return docs

    def valor(d, c):
        atual = d
        for parte in c.split("."):
            atual = atual.get(parte) if isinstance(atual, dict) else None
            if atual is None:
                return "—"
        if isinstance(atual, float):
            return f"{atual:,.2f}"
        if isinstance(atual, list):
            return ", ".join(map(str, atual))
        return str(atual)

    larg = [max(len(c), max(len(valor(d, c)) for d in docs)) for c in colunas]
    print("  " + "  ".join(c.ljust(w) for c, w in zip(colunas, larg)))
    print("  " + "  ".join("─" * w for w in larg))
    for d in docs:
        print("  " + "  ".join(valor(d, c).ljust(w) for c, w in zip(colunas, larg)))
    print(f"  [{len(docs)} documento(s)]\n")
    return docs


print("✅ Funções auxiliares: mostrar(...) e tabela(...)")

## 1. Documento vs relação

| | Relacional | Documento |
|---|-----------|-----------|
| Unidade | **Linha** em tabela | **Documento** (JSON/BSON) |
| Estrutura | Rígida, definida antes | Flexível, por documento |
| Relações | `JOIN` | Embutir ou referenciar |
| Dado relacionado | Espalhado em N tabelas | Frequentemente **junto** |
| Transação | ACID por padrão | ACID (desde a 4.0), com ressalvas |
| Consulta | SQL | API de documentos + pipeline |
| Nome disso | Tabela → linha → coluna | Coleção → documento → campo |

```
RELACIONAL                        DOCUMENTO
──────────                        ─────────
produtos                          {
┌────┬──────┬───────┐               "_id": 1,
│ id │ sku  │ preco │               "sku": "NB-01",
└────┴──────┴───────┘               "preco": 2599.90,
atributos                           "specs": {
┌───────┬───────┬──────┐               "ram_gb": 16,
│ p_id  │ chave │ valor│               "tela": 15.6
└───────┴───────┴──────┘             },
tags                                "tags": ["gamer", "leve"]
┌───────┬──────┐                  }
│ p_id  │ tag  │
└───────┴──────┘                  ← UM documento, UMA leitura
← 3 tabelas, 2 JOINs
```

### A ideia central

> **"Dados que são lidos juntos devem ficar juntos."**

No relacional você normaliza para evitar redundância e usa `JOIN` para remontar. No documento você **modela pela consulta**: se a tela do produto sempre mostra specs e tags, guarde tudo no mesmo documento e leia com uma operação.

## 2. ⚠️ Quando NÃO usar MongoDB

Esta seção vem **antes** do resto de propósito. "NoSQL" virou modismo por volta de 2012 e muita gente migrou sem precisar — e se arrependeu.

### 🔴 Não use quando

| Situação | Por quê |
|----------|---------|
| **Os dados são naturalmente relacionais** | Pedido → cliente → endereço → cidade. `$lookup` existe, mas é pior que `JOIN` |
| **Você precisa de relatórios ad-hoc** | O time de BI sabe SQL, não pipeline de agregação |
| **Integridade referencial importa** | Mongo **não tem chave estrangeira**. Nada impede um pedido órfão |
| **Transações entre entidades são o normal** | Funcionam, mas com custo e limitações |
| **A estrutura é estável** | Se todo documento tem os mesmos 12 campos, você quer uma tabela |
| **"É mais rápido"** | Postgres com índice correto costuma empatar ou ganhar |

### ✅ Use quando

| Situação | Por quê |
|----------|---------|
| **A estrutura varia de verdade** | Catálogo com atributos por categoria |
| **O documento é a unidade de acesso** | Sempre lê o objeto inteiro |
| **O schema evolui rápido** | Sem migração para adicionar campo |
| **Dado hierárquico e aninhado** | Árvore de comentários, configuração |
| **Volume alto com escrita simples** | Log, evento, telemetria |
| **Escala horizontal desde o dia 1** | Sharding nativo |

> 🧭 **Para a Aurora, a decisão é HÍBRIDA:**
>
> | Dado | Onde | Por quê |
> |------|------|---------|
> | Pedidos, itens, clientes, financeiro | **PostgreSQL** | Relacional, transacional, precisa de integridade |
> | Catálogo de produtos | **MongoDB** | Atributos variam por categoria, muda toda semana |
>
> Isso tem nome: **persistência poliglota**. Cada dado no banco que serve melhor a ele.
>
> ⚠️ **O custo é real:** dois bancos para operar, monitorar, fazer backup e manter consistentes entre si. Só vale quando o ganho compensa — e a pergunta honesta é: *o JSONB do Postgres não resolveria?* Em muitos casos, resolve.

## 3. BSON — o formato interno

O MongoDB guarda **BSON** (*Binary JSON*), que é JSON mais tipos.

| Tipo BSON | Python |
|-----------|--------|
| `Double`, `Int32`, `Int64` | `float`, `int` |
| `String` | `str` |
| `Boolean` | `bool` |
| `Array` | `list` |
| `Object` | `dict` |
| `Date` | `datetime` (⚠️ sempre UTC, milissegundos) |
| `ObjectId` | `bson.ObjectId` |
| `Null` | `None` |
| `Decimal128` | 💰 para dinheiro |

### `_id` — a chave primária obrigatória

Todo documento tem `_id`. Se você não fornecer, o Mongo gera um `ObjectId` de 12 bytes:

```
  507f1f77      bcf86c      d799      439011
  └timestamp┘   └máquina┘  └proc┘   └contador┘
```

> 💡 **O `ObjectId` carrega o instante de criação.** `objectid.generation_time` devolve a data — e como o prefixo é temporal, ordenar por `_id` ordena aproximadamente por data de criação.
>
> 💰 **Dinheiro:** o BSON `Double` é ponto flutuante, com o mesmo problema da aula 01_01. Para valores monetários, use `Decimal128` — ou guarde centavos como inteiro, como você fez no M03.

## 4. PyMongo — CRUD

```bash
pip install pymongo
```

```python
from pymongo import MongoClient

cliente = MongoClient("mongodb://localhost:27017")
db = cliente["atlas_catalogo"]      # banco
produtos = db["produtos"]           # coleção
```

> 💡 **Banco e coleção são criados na primeira escrita.** Não existe `CREATE DATABASE` nem `CREATE COLLECTION` obrigatório.

In [ ]:
from datetime import datetime, timezone

produtos = db["produtos"]

# Um documento
resultado = produtos.insert_one({
    "_id": "NB-DELL-15",                    # podemos usar o SKU como _id
    "nome": "Notebook Dell Inspiron 15",
    "categoria": "Notebooks",
    "preco": 2599.90,
    "custo": 2120.00,
    "estoque": 14,
    "ativo": True,
    "tags": ["informatica", "notebook", "trabalho"],
    "specs": {"ram_gb": 16, "tela_pol": 15.6, "ssd_gb": 512, "cor": "prata"},
    "criado_em": datetime(2026, 1, 15, tzinfo=timezone.utc),
})
print("inserido:", resultado.inserted_id)

In [ ]:
# Vários documentos — repare que cada um tem specs DIFERENTES
resultado = produtos.insert_many([
    {"_id": "NB-ACER-N5", "nome": "Notebook Acer Nitro 5", "categoria": "Notebooks",
     "preco": 3299.00, "custo": 2780.00, "estoque": 7, "ativo": True,
     "tags": ["informatica", "notebook", "gamer"],
     "specs": {"ram_gb": 32, "tela_pol": 15.6, "ssd_gb": 1024, "cor": "preto",
               "gpu": "RTX 3050", "taxa_hz": 144}},

    {"_id": "MO-LG-24UW", "nome": "Monitor LG 24 UltraWide", "categoria": "Monitores",
     "preco": 1199.00, "custo": 920.00, "estoque": 31, "ativo": True,
     "tags": ["informatica", "monitor"],
     "specs": {"tela_pol": 24, "resolucao": "2560x1080", "taxa_hz": 75,
               "painel": "IPS", "cor": "preto"}},

    {"_id": "CB-HDMI-2M", "nome": "Cabo HDMI 2.1 2m", "categoria": "Cabos",
     "preco": 69.90, "custo": 38.00, "estoque": 120, "ativo": True,
     "tags": ["cabo", "acessorio"],
     "specs": {"comprimento_m": 2, "versao": "2.1", "cor": "preto"}},

    {"_id": "CD-GAM-XT", "nome": "Cadeira Gamer XT Pro", "categoria": "Móveis",
     "preco": 1499.00, "custo": 980.00, "estoque": 5, "ativo": True,
     "tags": ["movel", "gamer", "ergonomia"],
     # ⬇️ Categoria NOVA, atributos que nunca existiram. Zero migração.
     "specs": {"altura_min_cm": 118, "altura_max_cm": 128, "peso_max_kg": 120,
               "espuma": "alta densidade", "reclina_graus": 180, "cor": "vermelho"}},

    {"_id": "FN-HYP-CL2", "nome": "Headset HyperX Cloud II", "categoria": "Áudio",
     "preco": 399.00, "custo": 255.00, "estoque": 29, "ativo": True,
     "tags": ["audio", "gamer", "headset"],
     "specs": {"impedancia_ohm": 60, "conexao": "USB/P2", "surround": "7.1",
               "peso_g": 320, "cor": "preto"}},

    {"_id": "PE-LOG-M170", "nome": "Mouse Logitech M170", "categoria": "Periféricos",
     "preco": 89.90, "custo": 52.00, "estoque": 240, "ativo": False,
     "tags": ["periferico", "mouse"],
     "specs": {"dpi": 1000, "conexao": "wireless", "cor": "cinza"}},
])

print(f"inseridos: {len(resultado.inserted_ids)}")
print(f"total na coleção: {produtos.count_documents({})}")

> 💭 **Repare no que acabou de acontecer.** A cadeira gamer entrou com `altura_min_cm`, `peso_max_kg` e `reclina_graus` — atributos que nenhum outro produto tem, e que não existiam quando a coleção foi criada.
>
> **Nenhum `ALTER TABLE`. Nenhuma migração. Nenhuma coluna nula.**
>
> É exatamente a dor do início da aula.

In [ ]:
# Leitura
tabela(produtos.find({}), ["_id", "nome", "categoria", "preco", "estoque"],
       "Todos os produtos")

In [ ]:
# find_one devolve UM documento (ou None)
mostrar([produtos.find_one({"_id": "CD-GAM-XT"})], "A cadeira gamer, inteira")

## 5. Consultas e operadores

| Operador | SQL equivalente | Exemplo |
|----------|-----------------|---------|
| `{"campo": v}` | `= v` | `{"categoria": "Notebooks"}` |
| `$eq` `$ne` | `=` `<>` | `{"preco": {"$ne": 100}}` |
| `$gt` `$gte` `$lt` `$lte` | `>` `>=` `<` `<=` | `{"preco": {"$gte": 1000}}` |
| `$in` `$nin` | `IN` `NOT IN` | `{"categoria": {"$in": ["A","B"]}}` |
| `$exists` | `IS NOT NULL` | `{"specs.gpu": {"$exists": True}}` |
| `$regex` | `LIKE` | `{"nome": {"$regex": "Note", "$options": "i"}}` |
| `$and` `$or` `$not` `$nor` | idem | `{"$or": [{...}, {...}]}` |
| `$all` | contém todos | `{"tags": {"$all": ["gamer","leve"]}}` |
| `$size` | tamanho do array | `{"tags": {"$size": 3}}` |
| `$elemMatch` | elemento que satisfaz | ver abaixo |

In [ ]:
tabela(produtos.find({"categoria": "Notebooks"}),
       ["_id", "nome", "preco"], "Igualdade simples")

tabela(produtos.find({"preco": {"$gte": 1000, "$lte": 3000}}),
       ["_id", "nome", "preco"], "Faixa de preço")

In [ ]:
# 🎯 Consulta em campo ANINHADO — a notação de ponto
tabela(produtos.find({"specs.cor": "preto"}),
       ["_id", "nome", "specs.cor"], "Produtos pretos (specs.cor)")

tabela(produtos.find({"specs.ram_gb": {"$gte": 16}}),
       ["_id", "nome", "specs.ram_gb"], "16 GB de RAM ou mais")

In [ ]:
# $exists — quais produtos TÊM determinado atributo
tabela(produtos.find({"specs.gpu": {"$exists": True}}),
       ["_id", "nome", "specs.gpu"], "Produtos com GPU")

tabela(produtos.find({"specs.peso_max_kg": {"$exists": True}}),
       ["_id", "nome", "specs.peso_max_kg"], "Produtos com peso máximo suportado")

> 💡 **`$exists` é a consulta que não existe no relacional.** Numa tabela, a coluna existe para todo mundo (mesmo que nula). Aqui, perguntar *"quais documentos têm este campo?"* é natural — e é a base de qualquer ferramenta de exploração de dados sem schema.

In [ ]:
# Arrays
tabela(produtos.find({"tags": "gamer"}), ["_id", "nome", "tags"],
       "Tem a tag 'gamer' (basta o valor, sem operador)")

tabela(produtos.find({"tags": {"$all": ["gamer", "ergonomia"]}}),
       ["_id", "nome", "tags"], "Tem TODAS: gamer E ergonomia")

tabela(produtos.find({"tags": {"$in": ["mouse", "headset"]}}),
       ["_id", "nome", "tags"], "Tem ALGUMA: mouse OU headset")

In [ ]:
# Lógicos e regex
consulta = {
    "$and": [
        {"ativo": True},
        {"$or": [
            {"categoria": "Notebooks"},
            {"preco": {"$lt": 100}},
        ]},
    ]
}
tabela(produtos.find(consulta), ["_id", "nome", "categoria", "preco", "ativo"],
       "Ativos que são notebooks OU custam menos de 100")

tabela(produtos.find({"nome": {"$regex": "gamer", "$options": "i"}}),
       ["_id", "nome"], "Nome contém 'gamer' (case-insensitive)")

In [ ]:
# Projeção — trazer só os campos necessários (o SELECT colunas do SQL)
mostrar(produtos.find(
    {"categoria": "Notebooks"},
    {"nome": 1, "preco": 1, "specs.ram_gb": 1, "_id": 0},   # 1 = incluir, 0 = excluir
), "Projeção: só o necessário")

> ⚠️ **Sempre projete em consulta de produção.** Sem projeção, o Mongo devolve o documento inteiro pela rede — e documentos podem ter megabytes. É o equivalente ao `SELECT *` que você aprendeu a evitar no M03.

In [ ]:
# Ordenação, limite e paginação
tabela(produtos.find({}, {"nome": 1, "preco": 1}).sort("preco", DESCENDING).limit(3),
       ["_id", "nome", "preco"], "Top 3 mais caros")

tabela(produtos.find({}, {"nome": 1, "preco": 1})
       .sort([("categoria", ASCENDING), ("preco", DESCENDING)]).skip(2).limit(3),
       ["_id", "nome", "preco"], "Ordenação múltipla, página 2")

## 6. Atualização parcial

Diferente do SQL, você **não reescreve o documento** — aplica operadores.

| Operador | Faz |
|----------|-----|
| `$set` | Define campo (cria se não existir) |
| `$unset` | Remove campo |
| `$inc` | Incrementa número |
| `$mul` | Multiplica |
| `$min` / `$max` | Atualiza só se for menor/maior |
| `$rename` | Renomeia campo |
| `$push` | Adiciona ao array |
| `$addToSet` | Adiciona **se não existir** |
| `$pull` | Remove do array por critério |
| `$pop` | Remove do início ou fim |
| `$currentDate` | Marca o instante atual |

In [ ]:
produtos.update_one(
    {"_id": "NB-DELL-15"},
    {"$set": {"preco": 2799.90, "specs.ram_gb": 32},   # ponto funciona no $set!
     "$inc": {"estoque": -3},
     "$addToSet": {"tags": "promocao"},
     "$currentDate": {"atualizado_em": True}},
)

mostrar([produtos.find_one({"_id": "NB-DELL-15"},
                           {"nome": 1, "preco": 1, "estoque": 1,
                            "tags": 1, "specs.ram_gb": 1, "atualizado_em": 1})],
        "Depois da atualização")

In [ ]:
# update_many — reajuste de 5% em uma categoria inteira
try:
    resultado = produtos.update_many(
        {"categoria": "Notebooks"},
        {"$mul": {"preco": 1.05}, "$set": {"reajustado": True}},
    )
    print(f"via $mul → correspondentes: {resultado.matched_count} | "
          f"modificados: {resultado.modified_count}")
except (ValueError, NotImplementedError) as erro:
    # ⚠️ O mongomock não implementa $mul. No MongoDB real, o bloco acima funciona.
    print(f"⚠️  $mul indisponível neste modo ({type(erro).__name__})")
    print("   Aplicando o reajuste em Python — o resultado é o mesmo.\n")
    for doc in produtos.find({"categoria": "Notebooks"}, {"preco": 1}):
        produtos.update_one(
            {"_id": doc["_id"]},
            {"$set": {"preco": round(doc["preco"] * 1.05, 2), "reajustado": True}},
        )

tabela(produtos.find({"categoria": "Notebooks"}), ["_id", "preco", "reajustado"],
       "Depois do reajuste")

In [ ]:
# Upsert — insere se não existir (o ON CONFLICT do M03)
resultado = produtos.update_one(
    {"_id": "TC-LOG-K380"},
    {"$set": {"nome": "Teclado Logitech K380", "categoria": "Periféricos",
              "preco": 279.00, "custo": 168.00, "estoque": 40, "ativo": True,
              "tags": ["periferico", "teclado", "bluetooth"],
              "specs": {"conexao": "bluetooth", "layout": "ABNT2", "cor": "branco"}}},
    upsert=True,
)
print("upserted_id:", resultado.upserted_id)

# Rodando de novo: agora ATUALIZA em vez de inserir
resultado = produtos.update_one(
    {"_id": "TC-LOG-K380"}, {"$set": {"preco": 299.00}}, upsert=True,
)
print("2ª execução → upserted_id:", resultado.upserted_id,
      "| modified:", resultado.modified_count)
print(f"\ntotal: {produtos.count_documents({})} documentos (não duplicou)")

In [ ]:
# $unset e $pull
produtos.update_one({"_id": "NB-DELL-15"},
                    {"$unset": {"reajustado": ""}, "$pull": {"tags": "promocao"}})
mostrar([produtos.find_one({"_id": "NB-DELL-15"}, {"nome": 1, "tags": 1, "reajustado": 1})],
        "Campo removido e tag retirada")

In [ ]:
# Remoção
db["temporaria"].insert_many([{"x": i} for i in range(5)])
print("antes :", db["temporaria"].count_documents({}))
db["temporaria"].delete_many({"x": {"$lt": 3}})
print("depois:", db["temporaria"].count_documents({}))
db.drop_collection("temporaria")

## 7. 🎯 Modelagem: embutir ou referenciar

**A decisão central do MongoDB.** Errar aqui é o que faz projetos NoSQL darem errado.

### Embutir (*embed*)

```javascript
{
  "_id": 1042,
  "cliente": {"nome": "Maria Souza", "email": "maria@x.com"},
  "itens": [
    {"sku": "NB-01", "nome": "Notebook", "qtd": 2, "preco": 2599.90},
    {"sku": "MO-01", "nome": "Monitor",  "qtd": 1, "preco": 1199.00}
  ]
}
```

### Referenciar (*reference*)

```javascript
// pedidos
{"_id": 1042, "cliente_id": 7, "itens": [{"produto_id": 1, "qtd": 2}]}
// clientes
{"_id": 7, "nome": "Maria Souza"}
```

### A tabela de decisão

| Embuta quando | Referencie quando |
|---------------|-------------------|
| Lido **junto**, sempre | Lido separadamente |
| Relação **1-poucos** | Relação 1-**muitos** ou N-N |
| O filho não existe sem o pai | O filho tem vida própria |
| O dado é **imutável depois** (histórico) | Muda com frequência |
| O documento cabe folgado em 16 MB | Cresce sem limite |

> 🔴 **O limite de 16 MB por documento é rígido.** Embutir uma coleção que cresce (comentários de um post viral, log de um pedido) leva a um erro que só aparece em produção, no documento que mais importa.
>
> ⚠️ **O padrão do array ilimitado é o erro nº 1 de modelagem em MongoDB.** Se você não consegue provar um teto para o tamanho da lista, **referencie**.

### Aplicando à Aurora

| Dado | Decisão | Por quê |
|------|---------|---------|
| `specs` dentro do produto | **Embutir** | Sempre lido junto, poucos campos, 1-1 |
| `tags` dentro do produto | **Embutir** | Array pequeno e limitado |
| Itens dentro do pedido | **Embutir** | Um pedido tem poucos itens, e eles não existem sem ele |
| Dados do cliente no pedido | **Embutir parcialmente** | Nome e e-mail no momento da compra — *snapshot*, como o `preco_unitario` do M03 |
| Pedidos dentro do cliente | 🔴 **Referenciar** | Cresce sem limite — 16 MB estourariam |
| Avaliações do produto | **Referenciar** | Podem ser milhares |

In [ ]:
# Modelagem híbrida na prática
pedidos = db["pedidos"]
pedidos.delete_many({})

pedidos.insert_many([
    {
        "_id": 1042,
        "data": datetime(2026, 7, 15, tzinfo=timezone.utc),
        "status": "pago",
        "canal": "site",
        # 📌 SNAPSHOT do cliente: nome e e-mail COMO ERAM na compra.
        #    Mesma ideia do preco_unitario do M03.
        "cliente": {"id": 7, "nome": "Maria Souza", "email": "maria@aurora.com.br",
                    "cidade": "Campinas", "uf": "SP"},
        # 📌 Itens EMBUTIDOS: poucos, imutáveis, sempre lidos junto
        "itens": [
            {"sku": "NB-DELL-15", "nome": "Notebook Dell Inspiron 15",
             "qtd": 2, "preco_unitario": 2599.90},
            {"sku": "MO-LG-24UW", "nome": "Monitor LG 24 UltraWide",
             "qtd": 1, "preco_unitario": 1199.00},
        ],
        "frete": 0.0,
    },
    {
        "_id": 1043, "data": datetime(2026, 7, 18, tzinfo=timezone.utc),
        "status": "pago", "canal": "app",
        "cliente": {"id": 12, "nome": "João Lima", "email": "joao@aurora.com.br",
                    "cidade": "São Paulo", "uf": "SP"},
        "itens": [{"sku": "CD-GAM-XT", "nome": "Cadeira Gamer XT Pro",
                   "qtd": 1, "preco_unitario": 1499.00}],
        "frete": 29.90,
    },
    {
        "_id": 1044, "data": datetime(2026, 8, 2, tzinfo=timezone.utc),
        "status": "cancelado", "canal": "marketplace",
        "cliente": {"id": 7, "nome": "Maria Souza", "email": "maria@aurora.com.br",
                    "cidade": "Campinas", "uf": "SP"},
        "itens": [{"sku": "FN-HYP-CL2", "nome": "Headset HyperX Cloud II",
                   "qtd": 3, "preco_unitario": 399.00}],
        "frete": 19.90,
    },
    {
        "_id": 1045, "data": datetime(2026, 8, 5, tzinfo=timezone.utc),
        "status": "pago", "canal": "site",
        "cliente": {"id": 23, "nome": "Ana Costa", "email": "ana@aurora.com.br",
                    "cidade": "Curitiba", "uf": "PR"},
        "itens": [
            {"sku": "NB-ACER-N5", "nome": "Notebook Acer Nitro 5",
             "qtd": 1, "preco_unitario": 3299.00},
            {"sku": "FN-HYP-CL2", "nome": "Headset HyperX Cloud II",
             "qtd": 1, "preco_unitario": 399.00},
            {"sku": "PE-LOG-M170", "nome": "Mouse Logitech M170",
             "qtd": 2, "preco_unitario": 89.90},
        ],
        "frete": 0.0,
    },
])

print(f"{pedidos.count_documents({})} pedidos inseridos")
mostrar([pedidos.find_one({"_id": 1042})], "Pedido completo em UMA leitura")

In [ ]:
# $elemMatch: pedidos que têm um item que satisfaz VÁRIAS condições ao mesmo tempo
tabela(pedidos.find({"itens": {"$elemMatch": {"qtd": {"$gte": 2}, "preco_unitario": {"$gt": 1000}}}}),
       ["_id", "cliente.nome", "status"],
       "Pedidos com item de 2+ unidades acima de R$ 1.000")

> ⚠️ **`$elemMatch` existe por um motivo sutil.** Sem ele, `{"itens.qtd": {"$gte": 2}, "itens.preco_unitario": {"$gt": 1000}}` casa se **um** item tem qtd ≥ 2 e **outro** tem preço > 1000 — não necessariamente o mesmo.
>
> É a diferença entre "existe item com A e existe item com B" e "existe item com A **e** B". Confundir os dois gera relatório errado.

## 8. 🎯 Pipeline de agregação

O `GROUP BY` do MongoDB — só que mais poderoso e mais verboso.

Um pipeline é uma **lista de estágios**, cada um transformando o resultado do anterior. É a mesma ideia do pipeline de geradores da aula 04_02.

| Estágio | SQL equivalente |
|---------|-----------------|
| `$match` | `WHERE` |
| `$group` | `GROUP BY` |
| `$sort` | `ORDER BY` |
| `$limit` / `$skip` | `LIMIT` / `OFFSET` |
| `$project` | `SELECT colunas` |
| `$addFields` | colunas calculadas |
| `$unwind` | 🎯 explode array em N documentos |
| `$lookup` | `LEFT JOIN` |
| `$facet` | várias agregações de uma vez |
| `$count` | `COUNT(*)` |

In [ ]:
# Contagem e média por categoria
resultado = produtos.aggregate([
    {"$match": {"ativo": True}},
    {"$group": {
        "_id": "$categoria",
        "produtos": {"$sum": 1},
        "preco_medio": {"$avg": "$preco"},
        "estoque_total": {"$sum": "$estoque"},
        "mais_caro": {"$max": "$preco"},
    }},
    {"$sort": {"preco_medio": -1}},
])
tabela(resultado, ["_id", "produtos", "preco_medio", "estoque_total", "mais_caro"],
       "Por categoria")

> ⚠️ **O `_id` do `$group` é a chave de agrupamento** — não o identificador do documento. É confuso no começo. Para agrupar por vários campos, use um objeto: `{"_id": {"cat": "$categoria", "cor": "$specs.cor"}}`.
>
> Para agregar **tudo**, use `{"_id": None}`.

In [ ]:
# 🎯 $unwind — o estágio mais importante para dados embutidos
# Cada item do array vira um documento próprio
mostrar(pedidos.aggregate([
    {"$match": {"_id": 1042}},
    {"$unwind": "$itens"},
    {"$project": {"_id": 1, "cliente": "$cliente.nome", "item": "$itens"}},
]), "$unwind: 1 pedido com 2 itens → 2 documentos")

In [ ]:
# Faturamento por produto — o relatório do M03, agora em pipeline
resultado = pedidos.aggregate([
    {"$match": {"status": "pago"}},
    {"$unwind": "$itens"},
    {"$group": {
        "_id": "$itens.sku",
        "nome": {"$first": "$itens.nome"},
        "unidades": {"$sum": "$itens.qtd"},
        "receita": {"$sum": {"$multiply": ["$itens.qtd", "$itens.preco_unitario"]}},
        "pedidos": {"$addToSet": "$_id"},
    }},
    {"$addFields": {"n_pedidos": {"$size": "$pedidos"}}},
    {"$project": {"pedidos": 0}},
    {"$sort": {"receita": -1}},
])
tabela(resultado, ["_id", "nome", "unidades", "receita", "n_pedidos"],
       "Ranking de produtos")

> 💡 **`$addToSet` + `$size` é o `COUNT(DISTINCT)` do Mongo.** Depois do `$unwind`, cada pedido aparece uma vez por item — exatamente a armadilha do `JOIN` que você viu no M03. Acumular os ids num conjunto e contar o tamanho resolve.
>
> **A armadilha é a mesma, em outra sintaxe.** Quem entendeu no SQL reconhece aqui.

In [ ]:
# Faturamento por cidade — agrupamento em DUAS ETAPAS
#
#   1ª: por pedido   → soma os itens, guarda o frete UMA vez ($first)
#   2ª: por cidade   → soma os pedidos
#
# É o mesmo cuidado do M03: depois de explodir os itens, o frete
# apareceria N vezes. O $first na primeira etapa resolve.
resultado = pedidos.aggregate([
    {"$match": {"status": "pago"}},
    {"$unwind": "$itens"},
    {"$group": {                                   # ── etapa 1: por pedido
        "_id": "$_id",
        "cidade": {"$first": "$cliente.cidade"},
        "uf": {"$first": "$cliente.uf"},
        "frete": {"$first": "$frete"},             # ⬅️ UMA vez, não N
        "total": {"$sum": {"$multiply": ["$itens.qtd", "$itens.preco_unitario"]}},
    }},
    {"$group": {                                   # ── etapa 2: por cidade
        "_id": {"cidade": "$cidade", "uf": "$uf"},
        "pedidos": {"$sum": 1},
        "receita": {"$sum": "$total"},
        "frete": {"$sum": "$frete"},
        "ticket_medio": {"$avg": "$total"},
    }},
    {"$project": {
        "_id": 0,
        "cidade": "$_id.cidade", "uf": "$_id.uf",
        "pedidos": 1, "receita": 1, "frete": 1, "ticket_medio": 1,
    }},
    {"$sort": {"receita": -1}},
])
tabela(resultado, ["cidade", "uf", "pedidos", "receita", "ticket_medio", "frete"],
       "Faturamento por praça")

> 💡 **O agrupamento em duas etapas é o idioma para "somar filhos sem inflar o pai".** Ele resolve, no Mongo, exatamente o problema que o `COUNT(DISTINCT)` resolvia no M03.
>
> 💭 **E existe um caminho ainda mais direto:** o operador `$map` aplica uma expressão a cada elemento do array **sem precisar de `$unwind`**:
>
> ```javascript
> {"$addFields": {"total": {"$sum": {"$map": {
>     "input": "$itens", "as": "i",
>     "in": {"$multiply": ["$$i.qtd", "$$i.preco_unitario"]}
> }}}}}
> ```
>
> ⚠️ **Mas não usamos aqui de propósito.** O `mongomock` implementa `$map` de forma incompleta e devolve **`0` silenciosamente** em vez de erro — o que produziria um relatório errado sem nenhum aviso.
>
> **Essa é uma lição sobre simulacros em geral:** eles cobrem o caminho comum e falham nas bordas, às vezes em silêncio. Se o seu código de produção depende de um recurso específico, **teste contra o serviço real** antes de confiar. Rode `docker compose up -d mongo` e experimente o `$map` acima.

In [ ]:
# $lookup — o LEFT JOIN do Mongo
resultado = pedidos.aggregate([
    {"$match": {"status": "pago"}},
    {"$unwind": "$itens"},
    {"$lookup": {
        "from": "produtos",
        "localField": "itens.sku",
        "foreignField": "_id",
        "as": "produto",
    }},
    {"$unwind": "$produto"},
    {"$project": {
        "_id": 0,
        "pedido": "$_id",
        "sku": "$itens.sku",
        "categoria": "$produto.categoria",
        "qtd": "$itens.qtd",
        "receita": {"$multiply": ["$itens.qtd", "$itens.preco_unitario"]},
        "custo_total": {"$multiply": ["$itens.qtd", "$produto.custo"]},
    }},
    {"$addFields": {"margem": {"$subtract": ["$receita", "$custo_total"]}}},
    {"$sort": {"margem": -1}},
])
tabela(resultado, ["pedido", "sku", "categoria", "qtd", "receita", "margem"],
       "$lookup: juntando pedidos e produtos")

> ⚠️ **`$lookup` é o `JOIN`, e é mais lento que o do relacional.**
>
> Ele foi adicionado na versão 3.2 justamente porque quem migrou descobriu que precisava juntar dados. **Se você usa `$lookup` o tempo todo, isso é um sinal de que seus dados são relacionais** — e talvez o banco errado tenha sido escolhido.
>
> Use com moderação, e sempre com índice no `foreignField`.

In [ ]:
# $facet — várias agregações independentes, numa passada só
resultado = list(pedidos.aggregate([
    {"$facet": {
        "por_status": [
            {"$group": {"_id": "$status", "n": {"$sum": 1}}},
            {"$sort": {"n": -1}},
        ],
        "por_canal": [
            {"$group": {"_id": "$canal", "n": {"$sum": 1}}},
            {"$sort": {"n": -1}},
        ],
        "totais": [
            {"$match": {"status": "pago"}},
            {"$unwind": "$itens"},
            {"$group": {"_id": None,
                        "receita": {"$sum": {"$multiply": ["$itens.qtd", "$itens.preco_unitario"]}},
                        "itens": {"$sum": "$itens.qtd"}}},
        ],
    }},
]))[0]

print("── por status ──")
for d in resultado["por_status"]:
    print(f"  {d['_id']:<12} {d['n']}")
print("\n── por canal ──")
for d in resultado["por_canal"]:
    print(f"  {d['_id']:<12} {d['n']}")
print("\n── totais (pagos) ──")
for d in resultado["totais"]:
    print(f"  receita R$ {d['receita']:,.2f} | {d['itens']} itens")

> 💡 **`$facet` é o que faz um dashboard em uma consulta.** No SQL você precisaria de várias consultas ou de CTEs com `UNION`. Aqui, cada faceta roda sobre o mesmo conjunto de entrada e devolve seu próprio resultado.

In [ ]:
# Analisando a variabilidade do schema — impossível no relacional
resultado = produtos.aggregate([
    {"$project": {"categoria": 1, "campos": {"$objectToArray": "$specs"}}},
    {"$unwind": "$campos"},
    {"$group": {
        "_id": "$campos.k",
        "produtos": {"$sum": 1},
        "categorias": {"$addToSet": "$categoria"},
    }},
    {"$addFields": {"n_categorias": {"$size": "$categorias"}}},
    {"$sort": {"produtos": -1, "_id": 1}},
])
tabela(resultado, ["_id", "produtos", "n_categorias", "categorias"],
       "Inventário de atributos: quem usa o quê")

> 💭 **Esta consulta é documentação gerada do próprio dado.** Num banco sem schema, ela é essencial: é como você descobre o que realmente existe na coleção, em vez de confiar num diagrama desatualizado.
>
> Guarde-a: é a primeira coisa a rodar ao herdar uma base MongoDB de outra pessoa.

## 9. Índices

Sem índice, o Mongo faz varredura completa da coleção — exatamente como o `Seq Scan` do M03.

In [ ]:
# Simples, composto e único
produtos.create_index("categoria")
produtos.create_index([("categoria", ASCENDING), ("preco", DESCENDING)])
produtos.create_index("specs.cor")
produtos.create_index("tags")            # índice multichave (arrays)
pedidos.create_index([("cliente.id", ASCENDING), ("data", DESCENDING)])
pedidos.create_index("status")

print("Índices em 'produtos':")
for indice in produtos.list_indexes():
    print(f"  {indice['name']:<28} {dict(indice['key'])}")

print("\nÍndices em 'pedidos':")
for indice in pedidos.list_indexes():
    print(f"  {indice['name']:<28} {dict(indice['key'])}")

### Tipos de índice

| Tipo | Para | Exemplo |
|------|------|---------|
| Simples | Um campo | `create_index("categoria")` |
| **Composto** | Vários campos | ⚠️ A ordem importa, igual ao M03 |
| **Multichave** | Arrays | Criado sozinho ao indexar um array |
| Texto | Busca em linguagem natural | `create_index([("nome", "text")])` |
| Geoespacial | Coordenadas | `2dsphere` |
| Hash | Sharding | `HASHED` |
| **TTL** | 🎯 Expiração automática | `expireAfterSeconds=3600` |
| Parcial | Subconjunto | `partialFilterExpression={"ativo": True}` |
| Único esparso | Único, ignorando ausentes | `unique=True, sparse=True` |

> 💡 **A regra do prefixo mais à esquerda vale igual ao M03.** Um índice em `(categoria, preco)` serve para consultas por `categoria` e por `categoria + preco` — **não** para `preco` sozinho.
>
> 🎯 **O índice TTL não tem equivalente simples no relacional.** Ele apaga documentos automaticamente depois de N segundos. É a forma padrão de implementar sessão, cache e retenção de log:
> ```python
> db.sessoes.create_index("criado_em", expireAfterSeconds=3600)
> ```

```javascript
// Analisando o plano — o EXPLAIN do Mongo
db.produtos.find({categoria: "Notebooks"}).explain("executionStats")

// O que olhar:
//   "stage": "COLLSCAN"  🔴 varreu a coleção inteira
//   "stage": "IXSCAN"    ✅ usou índice
//   totalDocsExamined  vs  nReturned
//     examinou 100.000 para devolver 12  →  falta índice
```

## 10. Transações e consistência

O MongoDB carregou por anos a fama de "não ter ACID". **Isso mudou.**

| Versão | O que ganhou |
|--------|--------------|
| Sempre | Atomicidade **por documento** |
| 4.0 | Transações **multi-documento** (replica set) |
| 4.2 | Transações distribuídas (cluster com sharding) |

```python
# 🍃 Exige MongoDB real com replica set — mongomock não suporta
with cliente.start_session() as sessao:
    with sessao.start_transaction():
        pedidos.insert_one({...}, session=sessao)
        produtos.update_one({"_id": sku}, {"$inc": {"estoque": -qtd}}, session=sessao)
    # commit automático ao sair sem exceção
```

> 💡 **A atomicidade por documento é mais poderosa do que parece.** Como os itens do pedido estão **embutidos**, criar um pedido com 5 itens é **uma única operação atômica** — sem transação nenhuma.
>
> No relacional, o mesmo caso exige um `INSERT` no pedido + 5 nos itens, dentro de uma transação. **A modelagem por documento reduz a necessidade de transação.**
>
> ⚠️ **Mas quando você precisa mesmo, há custo.** Transação multi-documento no Mongo é mais cara que no Postgres, exige replica set (não roda em instância única) e tem limite de tempo padrão de 60 segundos.

### Write concern e read preference

```python
from pymongo import WriteConcern, ReadPreference

# w=1        → confirma quando o primário gravou (padrão)
# w="majority" → confirma quando a maioria do replica set gravou (mais seguro)
# j=True     → só confirma após gravar no journal (durabilidade)
colecao = db.get_collection("pedidos", write_concern=WriteConcern(w="majority", j=True))
```

> ⚖️ **É um botão de ajuste entre desempenho e durabilidade.** `w=1` é rápido e pode perder escrita se o primário cair antes de replicar. `w="majority"` é mais lento e não perde. Para pedido financeiro, use `majority`; para telemetria, `w=1` basta.

## 🔧 Prática guiada — Catálogo poliglota

O desenho final para a Aurora: PostgreSQL para o transacional, MongoDB para o catálogo.

In [ ]:
# Simulando a chegada de uma linha de produto totalmente nova.
# No relacional isso seria uma migração; aqui é um insert.

novos = [
    {"_id": "IL-RGB-3M", "nome": "Fita LED RGB 3m", "categoria": "Iluminação",
     "preco": 129.90, "custo": 62.00, "estoque": 88, "ativo": True,
     "tags": ["iluminacao", "rgb", "setup"],
     "specs": {"comprimento_m": 3, "leds_por_m": 60, "controle": "app",
               "cores": 16000000, "tensao_v": 12}},

    {"_id": "SW-OFF-365", "nome": "Microsoft 365 Personal (1 ano)", "categoria": "Software",
     "preco": 349.00, "custo": 280.00, "estoque": 999, "ativo": True,
     "tags": ["software", "assinatura"],
     # 🎯 Produto DIGITAL: nem estoque físico nem dimensão fazem sentido.
     #    Atributos completamente diferentes de tudo que existia.
     "specs": {"tipo": "assinatura", "duracao_meses": 12, "dispositivos": 1,
               "armazenamento_nuvem_gb": 1024, "entrega": "digital"}},
]

produtos.insert_many(novos)
print(f"✅ {len(novos)} produtos de categorias INÉDITAS inseridos")
print(f"   Total: {produtos.count_documents({})}")
print("\n   Migração de schema necessária: nenhuma.")

tabela(produtos.find({"categoria": {"$in": ["Iluminação", "Software"]}}),
       ["_id", "nome", "categoria", "preco"], "As duas categorias novas")

In [ ]:
# Busca facetada — o filtro lateral de uma loja
def buscar(termo=None, categoria=None, preco_min=None, preco_max=None,
           tags=None, apenas_ativos=True, ordenar="preco", limite=10):
    """Monta a consulta dinamicamente, só com os filtros informados."""
    filtro = {}
    if apenas_ativos:
        filtro["ativo"] = True
    if termo:
        filtro["nome"] = {"$regex": termo, "$options": "i"}
    if categoria:
        filtro["categoria"] = categoria
    if preco_min is not None or preco_max is not None:
        faixa = {}
        if preco_min is not None:
            faixa["$gte"] = preco_min
        if preco_max is not None:
            faixa["$lte"] = preco_max
        filtro["preco"] = faixa
    if tags:
        filtro["tags"] = {"$all": tags}

    return filtro, list(
        produtos.find(filtro, {"nome": 1, "categoria": 1, "preco": 1, "tags": 1})
        .sort(ordenar, ASCENDING).limit(limite)
    )


filtro, resultado = buscar(preco_min=100, preco_max=1500, tags=["gamer"])
print("Filtro montado:")
print(json.dumps(filtro, ensure_ascii=False, indent=2))
print()
tabela(resultado, ["_id", "nome", "categoria", "preco"], "Gamer entre R$100 e R$1.500")

In [ ]:
# As facetas que a interface mostraria ao lado dos resultados
facetas = list(produtos.aggregate([
    {"$match": {"ativo": True}},
    {"$facet": {
        "categorias": [
            {"$group": {"_id": "$categoria", "n": {"$sum": 1}}},
            {"$sort": {"n": -1, "_id": 1}},
        ],
        "faixas_preco": [
            {"$bucket": {
                "groupBy": "$preco",
                "boundaries": [0, 100, 500, 1500, 5000],
                "default": "5000+",
                "output": {"n": {"$sum": 1}},
            }},
        ],
        "tags": [
            {"$unwind": "$tags"},
            {"$group": {"_id": "$tags", "n": {"$sum": 1}}},
            {"$sort": {"n": -1}},
            {"$limit": 8},
        ],
    }},
]))[0]

print("── Categorias ──")
for d in facetas["categorias"]:
    print(f"  {d['_id']:<16} ({d['n']})")

print("\n── Faixas de preço ──")
for d in facetas.get("faixas_preco", []):
    print(f"  a partir de {d['_id']:<8} ({d['n']})")

print("\n── Tags mais comuns ──")
for d in facetas["tags"]:
    print(f"  {d['_id']:<16} ({d['n']})")

In [ ]:
# Relatório final combinando as duas coleções
resultado = list(pedidos.aggregate([
    {"$match": {"status": "pago"}},
    {"$unwind": "$itens"},
    {"$lookup": {"from": "produtos", "localField": "itens.sku",
                 "foreignField": "_id", "as": "p"}},
    {"$unwind": "$p"},
    {"$group": {
        "_id": "$p.categoria",
        "unidades": {"$sum": "$itens.qtd"},
        "receita": {"$sum": {"$multiply": ["$itens.qtd", "$itens.preco_unitario"]}},
        "custo": {"$sum": {"$multiply": ["$itens.qtd", "$p.custo"]}},
        "pedidos": {"$addToSet": "$_id"},
    }},
    {"$addFields": {
        "margem": {"$subtract": ["$receita", "$custo"]},
        "n_pedidos": {"$size": "$pedidos"},
    }},
    {"$addFields": {
        "margem_pct": {"$cond": [
            {"$eq": ["$receita", 0]}, 0,
            {"$multiply": [{"$divide": ["$margem", "$receita"]}, 100]},
        ]},
    }},
    {"$project": {"pedidos": 0, "custo": 0}},
    {"$sort": {"receita": -1}},
]))

print(f"{'Categoria':<16}{'Pedidos':>9}{'Unid':>7}{'Receita':>13}{'Margem':>12}{'%':>8}")
print("─" * 65)
for d in resultado:
    print(f"{d['_id']:<16}{d['n_pedidos']:>9}{d['unidades']:>7}"
          f"{d['receita']:>13,.2f}{d['margem']:>12,.2f}{d['margem_pct']:>7.1f}%")

> 💭 **Compare com o M03.** A mesma pergunta de negócio, respondida em outro paradigma.
>
> O SQL era mais **legível**. O pipeline é mais **explícito** — cada estágio é uma transformação visível, e você consegue rodar o pipeline parcial para depurar (é só cortar a lista de estágios).
>
> **Nenhum dos dois é "melhor".** O que importa é: o dado que você tem se parece mais com uma tabela ou com um documento?

## 📝 Exercícios

**E1.** Liste três casos em que você **não** usaria MongoDB, com justificativa. Depois três em que usaria.

**E2.** Para a Aurora, argumente: o JSONB do PostgreSQL resolveria o problema do catálogo? O que se ganha e o que se perde ao usar o Mongo?

**E3.** Insira 5 produtos de categorias diferentes, cada um com `specs` completamente distintos. Depois liste todos os atributos existentes com sua frequência.

**E4.** Escreva consultas que encontrem: produtos entre R$500 e R$2.000; produtos com o atributo `gpu`; produtos com as tags `gamer` E `ergonomia`; produtos cujo nome contenha "pro" (ignorando caixa).

**E5.** Explique a diferença entre `{"itens.qtd": 2, "itens.preco": 100}` e `{"itens": {"$elemMatch": {"qtd": 2, "preco": 100}}}`. Crie dados que mostrem o resultado divergente.

**E6.** Use `update_many` com `$inc` para dar baixa no estoque de vários produtos. Depois use `$push` e `$pull` para gerenciar as tags.

**E7.** Implemente um upsert idempotente de catálogo: rodar 3 vezes com os mesmos dados não pode duplicar nada.

**E8.** Para cada caso, decida embutir ou referenciar, e justifique:
   - Endereços de entrega de um cliente (até 5)
   - Avaliações de um produto (podem ser milhares)
   - Itens de um pedido
   - Histórico de status de um pedido
   - Produtos de uma categoria

**E9.** Escreva um pipeline que responda: faturamento por mês, com nº de pedidos, ticket médio e variação em relação ao mês anterior.

**E10.** Use `$unwind` + `$group` + `$addToSet` para calcular quantos **pedidos distintos** contêm cada produto. Explique por que `$sum: 1` daria errado.

**E11.** Escreva um `$lookup` que junte pedidos e produtos e calcule a margem por categoria. Depois explique por que usar `$lookup` com frequência é um sinal de alerta.

**E12.** Use `$facet` para montar um dashboard com 4 métricas independentes numa consulta só.

**E13.** Crie índices para as consultas do E4. Justifique cada um e diga qual seria composto.

**E14.** Modele uma coleção `sessoes` com índice TTL de 30 minutos. Explique por que isso é melhor que um job de limpeza.

**E15.** Compare, escrevendo as duas versões: "faturamento por cidade" em SQL (M03) e em pipeline de agregação. Qual você acha mais legível? Qual é mais fácil de depurar?

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

## 📋 Cola de referência

```python
# ── Conexão ──
from pymongo import MongoClient, ASCENDING, DESCENDING
cliente = MongoClient("mongodb://localhost:27017")
db = cliente["atlas"]
col = db["produtos"]        # criado na primeira escrita

# ── Criar ──
col.insert_one({...})               .inserted_id
col.insert_many([{...}, {...}])     .inserted_ids

# ── Ler ──
col.find_one({"_id": x})
col.find(filtro, projecao).sort("campo", DESCENDING).skip(n).limit(m)
col.count_documents(filtro)
col.distinct("campo")

# projeção: {"campo": 1, "_id": 0}   1=incluir  0=excluir

# ── Operadores de consulta ──
{"c": v}                        # igualdade
{"c": {"$gte": 1, "$lte": 9}}   # faixa
{"c": {"$in": [a, b]}}          # pertence
{"c": {"$exists": True}}        # 🎯 campo existe
{"c": {"$regex": "x", "$options": "i"}}
{"$and": [...]}  {"$or": [...]}  {"$not": {...}}
{"tags": "x"}                   # array contém
{"tags": {"$all": [a, b]}}      # contém todos
{"tags": {"$size": 3}}
{"itens": {"$elemMatch": {"a": 1, "b": 2}}}   # ⚠️ MESMO elemento
{"specs.cor": "preto"}          # 🎯 notação de ponto

# ── Atualizar ──
col.update_one(filtro, {"$set": {...}, "$inc": {...}}, upsert=True)
col.update_many(filtro, {...})
# $set $unset $inc $mul $min $max $rename
# $push $addToSet $pull $pop  $currentDate

# ── Remover ──
col.delete_one(filtro)   col.delete_many(filtro)

# ── 🎯 Pipeline de agregação ──
col.aggregate([
    {"$match":  {...}},                    # WHERE (ponha PRIMEIRO)
    {"$unwind": "$array"},                 # explode array
    {"$lookup": {"from": "c", "localField": "a",
                 "foreignField": "_id", "as": "x"}},   # LEFT JOIN
    {"$group":  {"_id": "$campo",          # ⚠️ _id = chave do grupo
                 "n": {"$sum": 1},
                 "total": {"$sum": {"$multiply": ["$a", "$b"]}},
                 "media": {"$avg": "$x"},
                 "ids": {"$addToSet": "$_id"}}},       # COUNT(DISTINCT)
    {"$addFields": {"calc": {"$size": "$ids"}}},
    {"$project": {"_id": 0, "novo": "$antigo"}},
    {"$sort":   {"total": -1}},
    {"$limit":  10},
])

{"$facet": {"a": [...], "b": [...]}}       # várias agregações de uma vez
{"$bucket": {"groupBy": "$preco", "boundaries": [0,100,500]}}
{"$objectToArray": "$specs"}               # inspecionar schema
{"$map": {"input": "$arr", "as": "i", "in": "$$i.campo"}}
{"$cond": [condicao, se_sim, se_nao]}

# ── Índices ──
col.create_index("campo")
col.create_index([("a", ASCENDING), ("b", DESCENDING)])   # ⚠️ ordem importa
col.create_index("email", unique=True)
col.create_index("tags")                                   # multichave
col.create_index("criado_em", expireAfterSeconds=3600)     # 🎯 TTL
col.create_index([("nome", "text")])
col.list_indexes()   col.drop_index("nome_1")

# ── Transações (exige replica set) ──
with cliente.start_session() as s:
    with s.start_transaction():
        col.insert_one({...}, session=s)
```

```
🎯 EMBUTIR ou REFERENCIAR

  Embuta      lido junto · 1-poucos · imutável · cabe em 16 MB
  Referencie  lido separado · 1-muitos · muda muito · cresce sem limite

  🔴 Array sem teto = erro nº 1 de modelagem
```

## ✅ Checklist de saída

- [ ] Explico a diferença entre modelo de documento e relacional
- [ ] **Sei listar situações em que NÃO usar MongoDB**
- [ ] Sei argumentar se o JSONB do Postgres resolveria o caso
- [ ] Entendo persistência poliglota e seu custo operacional
- [ ] Sei o que é BSON e por que dinheiro pede `Decimal128`
- [ ] Sei que o `ObjectId` carrega o instante de criação
- [ ] Faço CRUD com PyMongo
- [ ] Uso notação de ponto para campos aninhados
- [ ] Conheço `$gte`, `$in`, `$exists`, `$regex`, `$all`, `$size`
- [ ] **Sei quando `$elemMatch` é obrigatório**
- [ ] Projeto os campos em consultas de produção
- [ ] Uso `$set`, `$inc`, `$push`, `$addToSet`, `$pull`
- [ ] Faço upsert idempotente
- [ ] 🎯 **Decido entre embutir e referenciar com critério**
- [ ] **Sei que array sem limite é o erro nº 1** (16 MB)
- [ ] Uso snapshot de dado histórico dentro do documento
- [ ] Monto pipelines com `$match`, `$unwind`, `$group`, `$sort`
- [ ] Sei que o `_id` do `$group` é a chave de agrupamento
- [ ] Uso `$addToSet` + `$size` como `COUNT(DISTINCT)`
- [ ] Sei que `$lookup` frequente é sinal de banco errado
- [ ] Uso `$facet` para dashboards
- [ ] Sei inspecionar o schema real com `$objectToArray`
- [ ] Crio índices, inclusive compostos e TTL
- [ ] Sei que o Mongo tem transações desde a 4.0, com ressalvas
- [ ] Entendo que embutir reduz a necessidade de transação

---

### ➡️ Próxima etapa

**`05_99_Lista_Exercicios.ipynb`** — Exercícios comparando ORM e NoSQL, e o projeto do módulo: migrar o Atlas para PostgreSQL com o catálogo no MongoDB.